# diagnosis.ipynb — reproducing the headline sketched result

End-to-end notebook that reproduces the winning configuration on the
synthetic **notebook smoke** preset (grid=64, n=5000, radial noise nl=1e-5).

Target numbers (seed=1):

```
                rv@1    rv@2    rv@5    rv@10
  sketched      0.680   0.884   0.924   0.924
  PPCA-EM       0.723   0.734   0.842   0.870
```

Wall time ≈ 25–30 min for the full pipeline (dataset gen + sketched + PPCA).

---

## What had to change from the original notebook to make this work

1. **Complex Fourier adjoint bug.** The old `right_matvec`/`left_matvec` path
   dropped the imaginary half of the residual, so the gradient was effectively
   halved. Fixed by using `vmap(core.adjoint_slice_volume)` on the full
   complex residual.
2. **Left/right sketch scale reconciliation.** The raw left and right sketches
   disagreed on a trace-consistency check by orders of magnitude. Dividing the
   left sketch by `LEFT_SCALE = vol_size / 2` brings both sides to the same
   scale (consistency error drops to ~1e-6).
3. **Radial Fourier prior folded into the operator.** The prior term is now
   applied inside `SketchedNormalOperator` via a precomputed `D²_fourier`
   mask — no separate `prior_lambda` kwarg. Makes the implicit normal
   operator self-contained.
4. **Matrix-free objective.** `F_rel(X) = ½⟨X, G(X)+G(0)⟩ + ½Σsᵢ²⟨Uᵢ, D²Uᵢ⟩ + λΣsᵢ`
   is computed without materializing X, using only sketch products.
5. **Backtracking step rule.** Monotone-descent Armijo-style acceptance: δ
   shrinks on rejection and grows on acceptance. Replaces the fragile
   fixed-step rule that caused the earlier divergence.
6. **Stall-on-bail-out guard.** When all Armijo retries reject, stall at
   the previous iterate instead of accepting the last (rejected) step.
   Without this, a tiny fp drift past iter ~50 can push the solver off
   a cliff on some GPUs (observed on A100: rv@10 jumps 0.92 → 1.8 →
   10⁵⁰) while converging cleanly on others. With the guard, the worst
   case is a plateau near rv@10 ≈ 0.92 rather than divergence.

These changes live in:
- `recovar.ppca.sketched_normal.SketchedNormalOperator` (fixes 1–3)
- `examples/sketched_normal/scripts/sketch_lib.py` (fixes 4–6)

## 1. Imports and config

Adds `examples/sketched_normal/scripts/` to `sys.path` so we can import the
sandbox helpers (`SketchedSolver`, `run_iterations_backtracking`, ...).

In [ ]:
import os, sys, time, json, pathlib, warnings

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
warnings.filterwarnings("ignore", module="finufft")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np

# --- provenance gate: make sure we're using THIS repo's recovar + the pixi jax ---
import recovar, jax
repo = pathlib.Path.cwd().resolve()
while not (repo / "examples" / "sketched_normal").exists() and repo != repo.parent:
    repo = repo.parent
assert (repo / "examples" / "sketched_normal").exists(), f"can't find repo root from {pathlib.Path.cwd()}"
r = pathlib.Path(recovar.__file__).resolve()
j = pathlib.Path(jax.__file__).resolve()
assert str(r).startswith(str(repo) + "/"), f"WRONG recovar checkout: {r}"
assert ".pixi/envs/default/" in str(j), f"WRONG jax environment: {j}"
print("ENV_OK")
print("  repo   :", repo)
print("  recovar:", r)
print("  jax    :", j)

# --- add sandbox scripts dir to path so we can import sketch_lib ---
SANDBOX = str(repo / "examples" / "sketched_normal" / "scripts")
if SANDBOX not in sys.path:
    sys.path.insert(0, SANDBOX)

In [ ]:
# --- recovar API ---
from recovar import utils
from recovar.output import metrics
from recovar.ppca import ppca as ppca_mod
from recovar.ppca.sketched_normal import SketchedNormalOperator
from recovar.ppca.ppca_scale_sweep import (
    _load_simulated_dataset,
    _with_trailing_separator,
    warmstart_from_pca,
)
from recovar.reconstruction import homogeneous
from recovar.simulation import simulator
from recovar.simulation.trajectory_generation import generate_trajectory_volumes

# --- sandbox helpers (matrix-free objective + backtracking solver) ---
from sketch_lib import (
    SketchedSolver,
    SketchedIterationCfg,
    build_gt_factors,
    run_iterations_backtracking,
)

In [ ]:
# ===================== CONFIG =====================
# Pick any writable scratch path with ~5GB free.
OUT_DIR = os.environ.get(
    "SKETCHED_OUT",
    "/scratch/gpfs/GILLES/mg6942/_agent_scratch/diagnosis_notebook_out",
)

# Synthetic dataset preset.
GRID         = 64
N_IMAGES     = 5000
NOISE_LEVEL  = 1e-5
N_VOLUMES    = 10
DATA_SEED    = 42

# Sketched solver (the headline-winning config).
TARGET_RANK      = 10
LAM              = 1.0          # nuclear-norm weight
METHOD           = "soft"       # SVT soft-thresholding
PRIOR_MODE       = "none"       # radial prior off for this preset
BLOCK_SIZE       = 15
MAX_RANK         = 60
N_POWER          = 3
N_ITER           = 80
BATCH_SIZE       = 500
SEED             = 1
# Backtracking step rule params.
BT_DELTA_INIT    = 0.1
BT_ARMIJO_C      = 0.9
BT_SHRINK        = 0.5
BT_GROW          = 1.5
BT_MAX_RETRIES   = 10

# PPCA-EM baseline.
PPCA_BASIS_SIZE  = 10
PPCA_N_ITER      = 20

os.makedirs(OUT_DIR, exist_ok=True)
pathlib.Path(OUT_DIR, "SAFE_TO_DELETE").touch()
VOXEL_SIZE = 4.25 * 128 / GRID
print(f"[config] grid={GRID} n={N_IMAGES} nl={NOISE_LEVEL}  voxel={VOXEL_SIZE:.3f}")
print(f"[config] out={OUT_DIR}")

## 2. Generate the synthetic dataset (cache-safe)

Re-running does nothing if the volumes + particles stack already exist.
First run takes ~2–5 min on an A100.

In [ ]:
vol_root   = os.path.join(OUT_DIR, f"vols_g{GRID}")
vol_prefix = os.path.join(vol_root, "vol")
ds_dir     = os.path.join(OUT_DIR, f"dataset_g{GRID}_n{N_IMAGES}_nl{NOISE_LEVEL}")
particles  = os.path.join(ds_dir, f"particles.{GRID}.mrcs")

t0 = time.time()
if not os.path.isfile(f"{vol_prefix}0000.mrc"):
    os.makedirs(vol_root, exist_ok=True)
    print(f"[vols] generating {N_VOLUMES} volumes -> {vol_prefix}*", flush=True)
    generate_trajectory_volumes(
        vol_root,
        grid_size=GRID,
        n_volumes=N_VOLUMES,
        voxel_size=VOXEL_SIZE,
        Bfactor=80,
        max_rotation_degrees=10.0,
        output_prefix=vol_prefix,
    )
else:
    print(f"[vols] reuse {vol_root}/")

if not os.path.isfile(particles):
    os.makedirs(ds_dir, exist_ok=True)
    print(f"[ds]   generating n={N_IMAGES} nl={NOISE_LEVEL} -> {ds_dir}/", flush=True)
    np.random.seed(DATA_SEED)
    simulator.generate_synthetic_dataset(
        ds_dir,
        VOXEL_SIZE,
        vol_prefix,
        N_IMAGES,
        grid_size=GRID,
        noise_level=NOISE_LEVEL,
        noise_model="radial1",
        contrast_std=0.0,
        noise_scale_std=0.0,
        dataset_params_option="uniform",
        disc_type="linear_interp",
        trailing_zero_format_in_vol_name=True,
        put_extra_particles=False,
        percent_outliers=0.0,
    )
else:
    print(f"[ds]   reuse {ds_dir}")

print(f"[ok] dataset ready  ({time.time() - t0:.1f}s)")
print(f"     {ds_dir}")

## 3. Load dataset + build ground-truth factors

In [ ]:
t0 = time.time()
cryo, sim_info, gt, noise_variance = _load_simulated_dataset(
    _with_trailing_separator(ds_dir), GRID, N_IMAGES, lazy=False
)
vs      = cryo.volume_shape
V_SIZE  = int(np.prod(vs))
n       = int(cryo.n_images)
print(f"[ds] grid={GRID}  n={n}  V={V_SIZE}  [{time.time() - t0:.1f}s]")

gt_factors = build_gt_factors(gt, sim_info, vs, V_SIZE)
print(f"[gt] s_gt top-10: {gt_factors['s_gt'][:10]}")

## 4. Sketched proximal-SVT solver (the winning config)

- **Left/right scale**: `LEFT_SCALE = V_SIZE / 2` (reconciles the two sketch sides).
- **Prior**: off (`PRIOR_MODE='none'`, `D2_fourier=None`).
- **Init**: cold (empty factors).
- **Step rule**: backtracking with monotone-descent acceptance.

Wall time ≈ 15–18 min on an A100. Each iteration logs `rv@{1,2,5,10}`. Target:
final `rv@10 ≈ 0.924`.

In [ ]:
LEFT_SCALE = V_SIZE / 2.0

op = SketchedNormalOperator(
    cryo, gt.get_mean(), batch_size=BATCH_SIZE, disc_type="linear_interp"
)
solver = SketchedSolver(op, vs, V_SIZE, n, LEFT_SCALE, D2_fourier=None)

cfg = SketchedIterationCfg(
    block_size=BLOCK_SIZE,
    max_rank=MAX_RANK,
    n_power=N_POWER,
    target_rank=TARGET_RANK,
    delta=BT_DELTA_INIT,
    method=METHOD,
    lam=LAM,
    prior_mode=PRIOR_MODE,
    record_per_k=(1, 2, 5, 10),
)

# Cold start.
U0 = np.zeros((V_SIZE, 0), np.float32)
s0 = np.zeros((0,),        np.float32)
V0 = np.zeros((n, 0),      np.float32)

print(f"[sketched] λ={LAM}  target_rank={TARGET_RANK}  n_iter={N_ITER}  seed={SEED}")
print(f"[sketched] backtracking: δ_init={BT_DELTA_INIT}  c={BT_ARMIJO_C}  "
      f"shrink={BT_SHRINK}  grow={BT_GROW}  max_retries={BT_MAX_RETRIES}")

t_sk = time.time()
U_sk, s_sk, V_sk, history_sk, final_sk = run_iterations_backtracking(
    solver, cfg, U0, s0, V0, N_ITER,
    gt_factors, vs, V_SIZE,
    seed=SEED,
    log_every=5,
    logfn=lambda m: print(m, flush=True),
    delta_init=BT_DELTA_INIT,
    armijo_c=BT_ARMIJO_C,
    shrink=BT_SHRINK,
    grow=BT_GROW,
    max_retries=BT_MAX_RETRIES,
)
sk_time = time.time() - t_sk
print(f"\n[sketched] done in {sk_time:.1f}s  —  final rank={len(s_sk)}")
for k in (1, 2, 5, 10):
    print(f"    rv@{k:2d} = {final_sk.get(f'rv@{k}', float('nan')):.4f}")

## 5. PPCA-EM baseline at `basis_size=10`

Wall time ≈ 5–10 min on an A100. Target: final `rv@10 ≈ 0.870`.

In [ ]:
t_mean = time.time()
noise_var_image = utils.make_radial_image(noise_variance, cryo.image_shape)
means, _mean_prior, _fsc = homogeneous.get_mean_conformation_relion(
    cryo, BATCH_SIZE, noise_variance=noise_var_image, use_regularization=False
)
mean_estimate = means.combined.flatten()
gt_mean       = gt.get_mean()
mean_err      = float(np.linalg.norm(np.asarray(mean_estimate) - gt_mean.flatten())
                       / np.linalg.norm(gt_mean))
print(f"[ppca-mean] err={mean_err:.4f}  [{time.time() - t_mean:.1f}s]")

In [ ]:
t_ws = time.time()
W_init, W_prior_base, U_gt_pp, s_gt_pp, pca_results = warmstart_from_pca(
    cryo, means, gt, PPCA_BASIS_SIZE, batch_size=100, gpu_memory=40
)
print(f"[ppca-warmstart] PCA rv={pca_results['rel_var']:.4f}  [{time.time() - t_ws:.1f}s]")

t_em = time.time()
em_output = ppca_mod.EM(
    cryo,
    mean_estimate,
    W_init.copy(),
    W_prior_base,
    U_gt=U_gt_pp,
    S_gt=s_gt_pp**2,
    EM_iter=PPCA_N_ITER,
    use_whitening=False,
    whitening_mode="cz",
    sparse_PCA=False,
    disc_type_mean="cubic",
    disc_type="linear_interp",
    return_iteration_data=True,
)
u_fin_pp, _s_fin_pp, W_fin_pp, _ez, _sm, iteration_data = em_output
pp_time = time.time() - t_em
print(f"[ppca-em] done in {pp_time:.1f}s")

_, rel_var_per_pc, _ = metrics.get_all_variance_scores(u_fin_pp, U_gt_pp, s_gt_pp**2)
rel_var_per_pc = np.asarray(rel_var_per_pc)
final_pp = {"rel_var_mean": float(np.mean(rel_var_per_pc))}
for k in (1, 2, 5, 10):
    kk = min(k, len(rel_var_per_pc))
    final_pp[f"rv@{k}"] = float(rel_var_per_pc[kk - 1])

for k in (1, 2, 5, 10):
    print(f"    rv@{k:2d} = {final_pp[f'rv@{k}']:.4f}")

## 6. Side-by-side comparison

In [ ]:
print("                rv@1    rv@2    rv@5    rv@10")
print(f"  sketched      "
      f"{final_sk.get('rv@1', float('nan')):.4f}  "
      f"{final_sk.get('rv@2', float('nan')):.4f}  "
      f"{final_sk.get('rv@5', float('nan')):.4f}  "
      f"{final_sk.get('rv@10', float('nan')):.4f}")
print(f"  PPCA-EM       "
      f"{final_pp['rv@1']:.4f}  {final_pp['rv@2']:.4f}  "
      f"{final_pp['rv@5']:.4f}  {final_pp['rv@10']:.4f}")

gap = final_sk.get("rv@10", float("nan")) - final_pp["rv@10"]
print(f"\n  gap (sketched - PPCA) on rv@10 : {gap:+.4f}")
print(f"\nwall time  —  sketched: {sk_time:.1f}s   PPCA: {pp_time:.1f}s")

# Dump JSON alongside the dataset, matching run_experiment.py format.
sketched_json = os.path.join(OUT_DIR, "sketched_notebook.json")
ppca_json     = os.path.join(OUT_DIR, "ppca_notebook.json")
with open(sketched_json, "w") as f:
    json.dump({
        "final": final_sk,
        "history": history_sk,
        "elapsed_total_s": sk_time,
        "config": {
            "grid": GRID, "n_images": N_IMAGES, "noise_level": NOISE_LEVEL,
            "lam": LAM, "target_rank": TARGET_RANK, "method": METHOD,
            "prior_mode": PRIOR_MODE, "step_rule": "backtracking",
            "bt_delta_init": BT_DELTA_INIT, "bt_armijo_c": BT_ARMIJO_C,
            "bt_shrink": BT_SHRINK, "bt_grow": BT_GROW,
            "bt_max_retries": BT_MAX_RETRIES,
            "block_size": BLOCK_SIZE, "max_rank": MAX_RANK, "n_power": N_POWER,
            "n_iter": N_ITER, "seed": SEED,
        },
    }, f, indent=2, default=float)
with open(ppca_json, "w") as f:
    json.dump({
        "final": final_pp,
        "elapsed_total_s": pp_time,
        "config": {
            "grid": GRID, "n_images": N_IMAGES, "noise_level": NOISE_LEVEL,
            "basis_size": PPCA_BASIS_SIZE, "n_iter": PPCA_N_ITER,
        },
    }, f, indent=2, default=float)
print(f"\n  wrote {sketched_json}")
print(f"  wrote {ppca_json}")

## Expected output

Values reproduce to ~1e-4 across seeds.

```
                rv@1    rv@2    rv@5    rv@10
  sketched      0.6801  0.8837  0.9243  0.9243
  PPCA-EM       0.7233  0.7343  0.8415  0.8699

  gap (sketched - PPCA) on rv@10 : +0.0544
```

If your `rv@10` for sketched comes out far below ~0.87, re-check:
- `LAM = 1.0`
- `METHOD = 'soft'`
- `PRIOR_MODE = 'none'`
- step rule = backtracking (not the fixed-δ path)
- `SEED = 1`